In [15]:
import numpy as np

In [3]:
import pandas as pd

In [4]:
df=pd.read_csv("/kaggle/input/datasets/kandeelai22/messy-e-commerce-sales-dataset/messy_ecommerce_sales_data.csv")

In [5]:
df.head()

,ID,Customer_Name,Order_ID,Order_Date,Product,Category,Quantity,Price,Payment_Method,Status,Total
0,100,Customer_100,ORD-41285,11/22/2024,Blender,Home,3,38,Cash on Delivery,Shipped,114.00
1,101,Customer_101,ORD-35783,7/5/2025,Smartphone,Electronics,2,abd,PayPal,Processing,NaN
2,102,Customer_102,ORD-84355,12/23/2024,Tennis Racket,Sports,1,389.05,PayPal,Delivered,389.05
3,103,Customer_103,ORD-57811,3/19/2025,Science,Books,5,233.92,PayPal,Processing,1169.60
4,104,Customer_104,ORD-93614,10/20/2025,Biography,Books,1,552.51,Cash on Delivery,Processing,552.51


In [6]:
df.shape

(103, 11)

In [7]:
df.columns.tolist()

['ID',
 ' Customer_Name',
 'Order_ID',
 'Order_Date',
 'Product',
 ' Category',
 'Quantity',
 'Price',
 'Payment_Method',
 'Status',
 'Total']

In [8]:
df.isnull().sum()

ID                 0
 Customer_Name     0
Order_ID           0
Order_Date         0
Product            0
 Category          8
Quantity           5
Price              5
Payment_Method     0
Status             0
Total             14
dtype: int64

In [9]:
# Raw columns have leading spaces: ' Customer_Name', ' Category'
df.columns = df.columns.str.strip()

# Verify
print(df.columns.tolist())

['ID', 'Customer_Name', 'Order_ID', 'Order_Date', 'Product', 'Category', 'Quantity', 'Price', 'Payment_Method', 'Status', 'Total']


In [10]:
# 1 exact duplicate found in audit
df = df.drop_duplicates(keep='first').reset_index(drop=True)

print(f"Rows after dedup: {len(df)}") 

Rows after dedup: 102


In [11]:
# Lowercase variation + all caps = same thing. Standardize to Title Case.
df['Category'] = df['Category'].str.strip().str.title()

# "Electronic" (missing 's') also needs fixing manually
df['Category'] = df['Category'].replace('Electronic', 'Electronics')

print(df['Category'].value_counts())

Category
Books          22
Home           20
Electronics    20
Sports         17
Clothing       15
Name: count, dtype: int64


In [16]:
# Remove stray $ signs first, then convert to number
df['Price'] = (
    df['Price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.strip()
)
df['Price'] = pd.to_numeric(df['Price'], errors='coerce')

# Business rule: Price must be between 0 and 5000
# Anything outside = data entry error → set to NaN (unknown)
df.loc[df['Price'] < 0, 'Price'] = np.nan
df.loc[df['Price'] > 5000, 'Price'] = np.nan

print(f"Price nulls after cleaning: {df['Price'].isna().sum()}")

Price nulls after cleaning: 10


In [17]:
# Convert to number — "4a" becomes NaN automatically
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

# Negative quantity = impossible (or ambiguous return) → flag it
df.loc[df['Quantity'] < 0, 'Quantity'] = np.nan

print(f"Quantity nulls after cleaning: {df['Quantity'].isna().sum()}")

Quantity nulls after cleaning: 8


In [18]:
# Recalculate from source — never trust a derived column in messy data
df['Total'] = (df['Quantity'] * df['Price']).round(2)

# How many rows now have a valid Total?
valid = df['Total'].notna().sum()
print(f"Rows with valid Total: {valid} / {len(df)}")

Rows with valid Total: 86 / 102


In [19]:
# pandas auto-detects "11/22/2024", "Jan 5 2023" etc.
# errors='coerce' turns "abc" → NaT instead of crashing
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

# Save the clean file
df.to_csv('ecommerce_cleaned.csv', index=False)
print("Saved! Ready for analysis.")

Saved! Ready for analysis.


In [20]:
df = pd.read_csv('/kaggle/working/ecommerce_cleaned.csv')
df['Order_Date'] = pd.to_datetime(df['Order_Date'], errors='coerce')

print("=== 1. REVENUE BY CATEGORY ===")
cat_rev = df.groupby('Category').agg(
    Orders=('Order_ID','count'),
    Revenue=('Total','sum'),
    Avg_Order_Value=('Total','mean')
).sort_values('Revenue', ascending=False).round(2)
print(cat_rev)

print("\n=== 2. ORDER STATUS BREAKDOWN ===")
status = df['Status'].value_counts()
status_pct = (status / len(df) * 100).round(1)
print(pd.DataFrame({'Count': status, 'Pct': status_pct}))

print("\n=== 3. REVENUE BY STATUS ===")
print(df.groupby('Status')['Total'].sum().sort_values(ascending=False).round(2))

print("\n=== 4. PAYMENT METHOD BREAKDOWN ===")
pay = df.groupby('Payment_Method').agg(
    Orders=('Order_ID','count'),
    Revenue=('Total','sum')
).sort_values('Revenue', ascending=False).round(2)
print(pay)

print("\n=== 5. CANCELLED ORDERS BY CATEGORY ===")
cancelled = df[df['Status']=='Cancelled'].groupby('Category').agg(
    Cancelled_Orders=('Order_ID','count'),
    Lost_Revenue=('Total','sum')
).sort_values('Lost_Revenue', ascending=False).round(2)
print(cancelled)

print("\n=== 6. MONTHLY REVENUE TREND ===")
df['Month'] = df['Order_Date'].dt.to_period('M')
monthly = df.groupby('Month')['Total'].sum().round(2)
print(monthly)

print("\n=== 7. TOP 5 PRODUCTS BY REVENUE ===")
top_products = df.groupby('Product')['Total'].sum().sort_values(ascending=False).head(5).round(2)
print(top_products)

print("\n=== 8. CANCELLATION RATE BY PAYMENT METHOD ===")
pay_cancel = df.groupby('Payment_Method').apply(
    lambda x: round((x['Status']=='Cancelled').sum() / len(x) * 100, 1)
).sort_values(ascending=False)
print(pay_cancel)

print("\n=== 9. AVG ORDER VALUE BY STATUS ===")
print(df.groupby('Status')['Total'].mean().round(2))


=== 1. REVENUE BY CATEGORY ===
             Orders   Revenue  Avg_Order_Value
Category                                      
Books            22  38357.11          1826.53
Clothing         15  27128.57          2466.23
Home             20  24729.71          1545.61
Electronics      20  20251.56          1125.09
Sports           17  17736.25          1364.33

=== 2. ORDER STATUS BREAKDOWN ===
            Count   Pct
Status                 
Returned       27  26.5
Processing     26  25.5
Shipped        23  22.5
Cancelled      15  14.7
Delivered      11  10.8

=== 3. REVENUE BY STATUS ===
Status
Returned      38477.22
Shipped       33487.73
Processing    25355.84
Cancelled     24944.04
Delivered     13006.70
Name: Total, dtype: float64

=== 4. PAYMENT METHOD BREAKDOWN ===
                  Orders   Revenue
Payment_Method                    
Cash on Delivery      33  42458.30
Bank Transfer         22  41612.83
Credit Card           23  26577.63
PayPal                24  24622.77

=== 5. CA

/tmp/ipykernel_58/2724220543.py:44: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pay_cancel = df.groupby('Payment_Method').apply(
